In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats

In [ ]:
import pickle

In [ ]:
from dask.distributed import Client

In [ ]:
# somehow without explicitly specifying 1 thread per worker, I get a lot of non-fatal errors out of xarray or netcdf4.
client = Client(n_workers=32, threads_per_worker=1)

In [ ]:
client

In [ ]:
times = np.arange(48000, 72001, 100)
files = [f"OUT_3D/BOMEX_r1_256_{t:010d}.nc" for t in times]
sam = xr.open_mfdataset(files, parallel=True, data_vars='all')
sam = sam.chunk({'time': 10})
# minutes = np.rint((sam.time - 173.0) * 24.0 * 60.0)
qt = sam.QV + sam.QN
wi = sam.W
# w = np.zeros_like(wi)
w = (wi.shift(z=-1, fill_value = 0.0)+ wi) * 0.5
qcl = sam.QN
tr = sam.TR01
qtm = qt.mean(axis=(0, 2, 3))
qts = qt.std(axis=(0, 2, 3))
wm = w.mean(axis=(0, 2, 3))
ws = w.std(axis=(0, 2, 3))
trm = tr.mean(axis=(0, 2, 3))
trs = tr.std(axis=(0, 2, 3))
tv = sam.TABS * (1.0 + 0.608 * sam.QV * 0.001 - sam.QN * 0.001)
tvm = tv.mean(axis=(0, 2, 3))
qclm = qcl.mean(axis=(0, 2, 3))

In [ ]:
import dask
import os

# Create the output directory if it doesn't exist
os.makedirs('pkl', exist_ok=True)

# Define all the variables to be computed
to_compute = {
    'wm': wm,
    'ws': ws,
    'trm': trm,
    'trs': trs,
    'qtm': qtm,
    'qts': qts,
    'tvm': tvm,
    'qclm': qclm,
}

# Compute all results in a single call
results = dask.compute(to_compute)[0]

# Save the results to pickle files
for name, data in results.items():
    with open(f'pkl/{name}.pkl', 'wb') as f:
        pickle.dump(data, f)